## Microsoft Entra ID 개요

Microsoft Entra ID(이전 Azure Active Directory)는 Microsoft의 클라우드 기반 ID 및 액세스 관리 서비스입니다. Microsoft 365, Azure 및 
수천 개의 기타 SaaS 애플리케이션을 위한 중앙 IdP 역할을 합니다.

주요 기능:
* **Single Sign-On(SSO)**: 한 번 인증하여 여러 애플리케이션에 액세스
* **Multi-Factor Authentication(MFA)**: 추가 검증 방법을 통한 보안 강화
* **Conditional Access**: 사용자, 디바이스, 위치, 위험을 기반으로 하는 정책 기반 액세스 제어
* **애플리케이션 통합**: OAuth 2.0, OpenID Connect, SAML 같은 최신 인증 프로토콜 지원

## 학습 목표
Microsoft Entra ID를 AgentCore Identity의 IdP로 사용하여 사용자를 인증하고, 에이전트가 사용자를 대신해 보호된 리소스에 액세스하도록 권한을 부여할 수 있습니다. 

<img src="images/entra-notebook-overview.png" width="75%">

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.

* Python 3.10+
* AWS 자격 증명
* Strands Agents
* 설치된 Docker, Finch 또는 Podman
* AWS 리전을 "us-west-2" 또는 Bedrock AgentCore를 지원하는 리전으로 설정. 지원 리전은 https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html 참조

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## 1단계: Entra ID Tenant 설정

## Authorization Code Flow
OAuth 2.0 authorization code flow는 웹 애플리케이션에서 사용자를 안전하게 인증하고 access token을 얻기 위한 권장 방식입니다. 이 
흐름은 다음 단계로 구성됩니다.
1. 인증을 위해 사용자를 Entra ID로 리디렉션
2. 로그인 성공 후 authorization code 수신
3. code를 access token 및 refresh token으로 교환
4. 토큰을 사용해 보호된 리소스에 액세스

이 통합 pattern을 사용하면 애플리케이션에 안전한 표준 기반 인증을 유지하면서 AgentCore에서 Entra ID의 강력한 ID 관리 기능을 활용할 수 있습니다.

Entra ID tenant는 조직을 나타내는 전용 Microsoft Entra ID 인스턴스입니다. Microsoft 클라우드에 격리된 조직 디렉터리라고 생각할 수 있습니다.

주요 특성:
* **고유한 ID**: 각 tenant에는 고유한 domain이 있습니다(예: yourcompany.onmicrosoft.com).
* **격리된 경계**: 한 tenant의 사용자, 그룹, 애플리케이션은 다른 tenant와 분리됩니다.
* **관리 제어**: tenant 관리자가 사용자, 보안 정책, app registration을 관리합니다.
* **다중 Domain 지원**: 기본 .onmicrosoft.com domain과 함께 custom domain을 포함할 수 있습니다.

실제 적용:

OAuth 통합을 위해 Entra ID에 애플리케이션을 등록하면 특정 tenant 내부에 등록됩니다. 이후 해당 tenant의 사용자는 조직 자격 증명으로 애플리케이션에 인증할 수 있습니다.

AgentCore 통합에는 다음 항목이 필요합니다.
* **Tenant ID**: Entra ID 인스턴스의 고유 식별자
* **Application Registration**: tenant에 등록된 앱
* **적절한 권한**: 애플리케이션에 구성된 액세스 권한

이 tenant 기반 모델은 인증과 권한 부여가 조직의 보안 경계 안에서 유지되도록 합니다.

tenant 생성 단계는 https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant 에서 확인할 수 있습니다.

참고:
1. Microsoft Entra ID는 AWS 서비스가 아닙니다. 비용 관련 정보는 Microsoft Entra ID 문서를 참조하세요.
2. 다음 단계의 화면은 변경될 수 있습니다. Entra ID 애플리케이션 설정에 대한 최신 지침은 Microsoft Entra ID 문서를 참조하세요.

## 2단계: 애플리케이션 설정

1. https://portal.azure.com 으로 이동하여 화면 상단의 검색창에서 "Entra ID"를 검색합니다.
<img src="images/entraid.jpg" width="75%">

2. `Manage` &rarr; `App Registrations`로 이동합니다.
<img src="images/app.registration.png" width="75%">


3. `New Registration`을 클릭하고 세부 정보를 입력합니다. multi-tenant 옵션을 선택하세요.
<img src="images/app.registration.form.png" width="75%">


4. client secret을 생성합니다. AgentCore Identity에서 사용할 clientId와 client secret을 복사합니다.
<img src="images/gather.client.info.png" width="75%">


5. OAuth scope를 생성합니다. Expose an API &rarr; `Add Scope`로 이동하여 전체 scope를 복사해 저장합니다. 
<img src="images/expose.api.png" width="75%">


6. OneNote 액세스를 허용하도록 API 권한을 추가합니다. "API Permissions" --> "Add a permission" --> "Microsoft API" --> "OneNote" --> "Delegated Permissions"
<img src="images/onenote.api.perm.png" width="75%"/>

## 2단계 - Bedrock AgentCore Identity Provider 생성

1단계에서 확인한 tenant 및 애플리케이션 정보로 아래 환경 변수를 업데이트하세요.

다음 값을 확인하세요.
- Tenant ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Directory (tenant) ID"
- Client ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Application (client) ID"
- 앞 단계에서 저장한 secret
- Scope: "openid profile https://graph.microsoft.com/Notes.ReadWrite.All https://graph.microsoft.com/Notes.Create"
- Audience: "https://graph.microsoft.com"

In [ ]:
import os

# client_id로 교체
os.environ["client_id"] = "REPLACE_ME"

# secret으로 교체
os.environ["secret"] = "REPLACE_ME"

# tenant_id로 교체
os.environ["tenant_id"] = "REPLACE_ME"

##########

# 필요한 경우 scope로 교체
os.environ["scopes"] = (
    "openid profile https://graph.microsoft.com/Notes.ReadWrite.All https://graph.microsoft.com/Notes.Create"
)

# 필요한 경우 audience로 교체
os.environ["audience"] = "https://graph.microsoft.com"

Amazon Bedrock AgentCore Identity는 Inbound Auth와 Outbound Auth 모두에 관리형 OAuth 2.0 지원 provider를 제공합니다. 

에이전트에서 사용할 IdP를 생성합니다. provider는 서로 다른 OAuth 2.0 구현, API 인증 체계, 토큰 형식의 복잡성을 추상화하고, 내부 프로토콜 차이와 예외 상황을 처리하면서 에이전트에 일관된 인터페이스를 제공합니다.

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient
from boto3.session import Session


boto_session = Session()
region = boto_session.region_name

if not region:
    import warnings

    warnings.warn("There is no configured Region in the AWS session. Defaulting to us-east-1")
    region = "us-east-1"

identity_client = IdentityClient(region=region)

ms_provider = identity_client.create_oauth2_credential_provider(
    req={
        "name": "microsoft_entra_oauth_provider",
        "credentialProviderVendor": "MicrosoftOauth2",
        "oauth2ProviderConfigInput": {
            "microsoftOauth2ProviderConfig": {
                "clientId": os.environ["client_id"],
                "clientSecret": os.environ["secret"],
                "tenantId": os.environ["tenant_id"],
            }
        },
    }
)
print(f"Microsoft Credential Provider: {ms_provider}")
print()
print(f"Callback URL: {ms_provider['callbackUrl']}")

## 2.5단계: Credential Provider의 Callback URL로 Microsoft Entra 업데이트

<img src="images/redirect.uri.png" width="75%">

발급할 Token을 선택합니다.

<img src="images/select.tokens.jpg" width="75%">

## 3단계: 로컬 검증

AgentCore Identity를 사용하면 구성된 OAuth 2.0 credential provider를 기반으로 사용자 위임 액세스 또는 machine-to-machine 인증용 OAuth 토큰을 얻을 수 있습니다. 

이 서비스는 사용자 또는 애플리케이션과 downstream authorization server 사이의 인증 과정을 오케스트레이션하고 결과 토큰을 가져와 저장합니다. AgentCore Identity vault에서 토큰을 사용할 수 있게 되면 권한이 있는 에이전트가 토큰을 가져와 resource server 호출 권한 부여에 사용할 수 있습니다. 

##### 아래 코드에서는 사용자 위임 흐름에 Entra ID를 사용합니다.

In [ ]:
from bedrock_agentcore.identity.auth import requires_access_token
from oauth2_callback_server import get_oauth2_callback_url


@requires_access_token(
    provider_name="microsoft_entra_oauth_provider",
    auth_flow="USER_FEDERATION",
    scopes=os.environ["scopes"].split(" "),
    on_auth_url=lambda x: print("\nPlease copy and paste this URL in your browser:\n" + x),
    force_authentication=True,
    callback_url=get_oauth2_callback_url(),
)
def need_access_token(*, access_token: str):
    return access_token

##### `need_access_token(access_token="")`을 실행하면 Entra ID에 인증하고 애플리케이션이 사용할 권한 부여 토큰을 얻기 위한 URL이 표시됩니다. 인증하고 동의하면 authorization code를 사용할 수 있습니다. 

<img src="images/authenticate.and.authorize.png" width="75%">


In [ ]:
import sys
import subprocess

from oauth2_callback_server import wait_for_oauth2_server_to_be_ready

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

id_token = ""
try:
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        id_token = need_access_token(access_token="")
        print(f"Bearer Token Received: {id_token[:10]}...")
finally:
    oauth2_callback_server_process.terminate()

##### 토큰을 decode하여 로컬에서 검증할 수 있습니다.

In [ ]:
import json
import jwt  # PyJWT 라이브러리

# 검사 목적으로만 검증 없이 토큰 decode
# 프로덕션에서는 항상 토큰의 signature와 claim을 검증
decoded_token = jwt.decode(id_token, options={"verify_signature": False})

print(f"Decoded Bearer Token (for inspection): \n{json.dumps(decoded_token, indent=4)}")

##### Entra ID에서 받은 decoded token은 아래와 유사합니다.
<img src="images/decoded-token.png" width="75%">

## 4단계 - AgentCore Runtime 에이전트로 통합

### OneNote 통합 에이전트

이 코드는 사용자가 자연어 명령으로 Microsoft OneNote Notebook을 생성하고 관리하도록 돕는 AI 에이전트를 만듭니다. 에이전트는 Entra ID 인증으로 OneNote API에 액세스하고 다음 세 가지 주요 기능을 제공합니다.

1. Notebook 생성: 새 OneNote Notebook 생성(`create_notebook` 도구)
2. Section 생성: 기존 Notebook에 section 추가(`create_notebook_section` 도구)
3. 콘텐츠 추가: Notebook section에 콘텐츠가 포함된 페이지 생성(`add_content_to_notebook_section` 도구)

에이전트는 OAuth2 인증을 자동으로 처리하며 필요할 때 사용자에게 권한 부여를 요청한 다음, 회의 메모나 기타 콘텐츠를 구조화된 OneNote Notebook으로
정리하도록 사용자 요청을 처리합니다.

In [ ]:
%%writefile strands_entraid_onenote.py
import os
import json
import asyncio
import requests

from strands import Agent
from strands import tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token
from strands.models.bedrock import BedrockModel
from oauth2_callback_server import get_oauth2_callback_url

os.environ["STRANDS_OTEL_ENABLE_CONSOLE_EXPORT"] = "true"
os.environ["OTEL_PYTHON_EXCLUDED_URLS"] = "/ping,/invocations"

entra_access_token = None  # access token을 저장할 전역 변수
tool_name = None

@tool
def create_notebook(name: str) -> str:
    """
    Create a new Microsoft OneNote notebook for the user. Needed before you can create a section or add content.
    
    Args:
        name (str): The display name for the new notebook
        
    Returns:
        str: The ID of the created notebook
    """
    global entra_access_token
    global tool_name 
    tool_name = "create_notebook"
    # 이미 토큰이 있는지 확인
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'application/json'
    }
    # 새 Notebook 생성
    notebook_data = {'displayName': name}
    notebook = requests.post(
        'https://graph.microsoft.com/v1.0/me/onenote/notebooks', 
        headers=headers, 
        json=notebook_data
    )
    notebook.raise_for_status()
    return json.dumps({"notebook_id": notebook.json()['id']})

@tool
def create_notebook_section(notebook_id: str, section_name: str) -> str:
    """
    Create a new section in an existing OneNote notebook. Section is created for a specific notebook. 
    
    Args:
        notebook_id (str): The ID of the OneNote notebook to create the section in
        section_name (str): The display name for the new section
        
    Returns:
        str: The ID of the created section
    """
    global entra_access_token
    global tool_name 
    tool_name = "create_notebook_section"
    # 이미 토큰이 있는지 확인
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'application/json'
    }
    # 새 section 생성
    section_data = {'displayName': section_name}
    section = requests.post(
        f'https://graph.microsoft.com/v1.0/me/onenote/notebooks/{notebook_id}/sections',
        headers=headers, 
        json=section_data
    )
    section.raise_for_status()
    
    section_id = section.json()['id']
    return json.dumps({"section_id": section_id})

@tool
def add_content_to_notebook_section(section_id: str, page_content) -> str:
    """
    Add content to a OneNote notebook section by creating a new page.
    
    Args:
        section_id (str): The ID of the OneNote section to add content to
        page_content: The HTML content to add as a new page
        
    Returns:
        str: URL to the created notebook page
    """
    global entra_access_token
    global tool_name 
    tool_name = "add_content_to_notebook_section"
    
    # 이미 토큰이 있는지 확인
    if not entra_access_token:
        return json.dumps({"auth_required": True, "message": f"Entra ID authentication is required for {tool_name}. Please wait while we set up the authorization.", "events": []})

    headers = {
        'Authorization': f'Bearer {entra_access_token}',
        'Content-Type': 'text/html'
    }
    page = requests.post(
        f'https://graph.microsoft.com/v1.0/me/onenote/sections/{section_id}/pages',
        headers=headers, 
        data=page_content
    )
    page.raise_for_status()
    url = json.loads(page.text)["links"]["oneNoteWebUrl"]["href"]
    return json.dumps({"oneNoteWebUrl": url})
    
    
# 도구와 함께 에이전트 초기화
model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")
system_prompt = """You are an Agent who helps user put in their meeting into OneNote notebooks. 
    Identify the notebook name, section name and content based on what the user has provided. 
    Return notebook URL once created."""
agent = Agent(model=model, system_prompt=system_prompt, tools=[create_notebook, create_notebook_section, add_content_to_notebook_section])

# 앱 및 스트리밍 queue 초기화
app = BedrockAgentCoreApp()

class StreamingQueue:
    def __init__(self):
        self.finished = False
        self.queue = asyncio.Queue()
        
    async def put(self, item):
        await self.queue.put(item)

    async def finish(self):
        self.finished = True
        await self.queue.put(None)

    async def stream(self):
        while True:
            item = await self.queue.get()
            if item is None and self.finished:
                break
            yield item

queue = StreamingQueue()

async def on_auth_url(url: str):
    print(f"Authorization url: {url}")
    await queue.put(f"Authorization url: {url}")


async def agent_task(user_message: str):
    global tool_name
    try:
        await queue.put("Begin agent execution")
        
        # 인증 필요 여부를 확인하도록 먼저 에이전트 호출
        response = agent(user_message)
        
        # 응답 구조에서 텍스트 콘텐츠 추출
        response_text = ""
        if isinstance(response.message, dict):
            content = response.message.get('content', [])
            if isinstance(content, list):
                for item in content:
                    if isinstance(item, dict) and 'text' in item:
                        response_text += item['text']
        else:
            response_text = str(response.message)
        
        # 응답에 인증 필요가 표시되는지 확인
        # 인증 문제를 나타내는 여러 keyword 확인
        auth_keywords = [
            "authentication", "authorize", "authorization", "auth", 
            "sign in", "login", "access", "permission", "credential",
            "need authentication", "requires authentication"
        ]
        needs_auth = any(keyword.lower() in response_text.lower() for keyword in auth_keywords)
       
        if needs_auth:
            await queue.put(f"Authentication required for {tool_name} access. Starting authorization flow...")
            
            # 3LO 인증 흐름 시작
            try:
                global entra_access_token
                entra_access_token = await need_token_3LO_async(access_token=None)
                await queue.put(f"Authentication successful! Retrying {tool_name}...")
                
                # 인증을 완료했으므로 에이전트 호출 재시도
                response = agent(user_message)
            except Exception as auth_error:
                print(f"auth_error: ", repr(auth_error))
                await queue.put(f"Authentication failed: {repr(auth_error)}")
        
        await queue.put(response.message)
        await queue.put("End agent execution")
    except Exception as e:
        await queue.put(f"Error: {repr(e)}")
    finally:
        await queue.finish()

@requires_access_token(
    provider_name="microsoft_entra_oauth_provider",
    scopes=os.environ["scopes"].split(' '),
    auth_flow='USER_FEDERATION',
    on_auth_url=on_auth_url,
    force_authentication=True,
    callback_url=get_oauth2_callback_url(),
)
async def need_token_3LO_async(*, access_token: str):
    global entra_access_token
    entra_access_token = access_token  # 전역 access token 업데이트
    print("Got access token....", access_token)
    return access_token


@app.entrypoint
async def agent_invocation(payload):
    user_message = payload.get("prompt", "No prompt found in input, please guide customer to create a json payload with prompt key")
    
    # 에이전트 task 생성 및 시작
    task = asyncio.create_task(agent_task(user_message))

    # task가 동시에 실행되도록 보장하면서 stream 반환
    async def stream_with_task():
        # 결과가 생성되는 즉시 스트리밍
        async for item in queue.stream():
            yield item
        
        # task 완료 보장
        await task
    
    return stream_with_task()
    
if __name__ == "__main__":
    app.run()


##### AgentCore Runtime 구성

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_entraid_onenote.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_entraid_onenote_3lo",
)

print(f"Agent Configure Response: {response}")

##### 에이전트 시작. 시작 후 애플리케이션에서 에이전트를 사용할 수 있습니다.

In [ ]:
launch_response = agentcore_runtime.launch(
    local_build=False,
    auto_update_on_conflict=True,
    env_vars={
        "scopes": os.environ["scopes"],
    },
)

print(f"Launch Response: {launch_response}")

#### 아래 코드 셀에서 URL이 제공됩니다. 브라우저 창에서 인증하려면 아래 이미지의 URL이 아니라 실제로 받은 URL을 복사하세요.
<img src="images/url.presented.png" width="75%">

#### 인증 요청이 표시되면 인증을 완료하세요.
<img src="images/authenticate.and.authorize.png" width="75%">

#### "Bedrock Agents"라는 Notebook이 이미 있으면 반드시 삭제하세요.

In [ ]:
import sys
import uuid
import subprocess

from typing import Final
from oauth2_callback_server import (
    store_user_id_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
)


prompt = """
Put these notes into onenote notebook named "Bedrock Agents".

Amazon Bedrock AgentCore enables you to deploy and operate 
highly capable AI agents securely, at scale. It offers 
infrastructure purpose-built for dynamic agent workloads, 
powerful tools to enhance agents, and essential controls for 
real-world deployment. AgentCore services can be used 
 together or independently and work with any framework including 
CrewAI, LangGraph, LlamaIndex, and Strands Agents, as well as 
any foundation model in or outside of Amazon Bedrock, giving you 
ultimate flexibility. AgentCore eliminates the undifferentiated 
heavy lifting of building specialized agent infrastructure, so 
you can accelerate agents to production. Provide link to 
the created OneNote Notebook and provide error message from the API 
in case of failure.
"""
session_id = str(uuid.uuid1())
user_id: Final[str] = "user"

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        store_user_id_in_oauth2_callback_server(user_id)
        st = agentcore_runtime.invoke(payload={"prompt": prompt}, session_id=session_id, user_id=user_id)
finally:
    oauth2_callback_server_process.terminate()

## 4단계 - 생성된 OneNote Notebook 검증
- 위 에이전트 호출 함수는 Notebook과 그 안의 section을 생성하고 section에 콘텐츠를 추가합니다.
- 사용자는 https://your-domain-name-my.sharepoint.com/ 에 로그인하여 생성된 Notebook에 액세스할 수 있습니다.

domain 이름은 다음 화면에서 확인할 수 있습니다.

<img src="images/domain.name.png" width="75%"/>

로그인하면 SharePoint 홈 페이지로 이동합니다. 

<img src="images/sharepoint.home.png" width="75%"/>

My Files --> Notebooks로 이동합니다.

<img src="images/sharepoint.my.files.png" width="75%"/>

## 마무리 및 정리

이 Notebook에서는 다음 내용을 알아보았습니다.
- OAuth Authorization Code flow를 제공하도록 Entra ID API와 애플리케이션 설정
- AgentCore Runtime을 생성하고 사용자를 대신해 OneNote Notebook을 생성하는 도구가 포함된 에이전트 배포

#### 생성된 리소스

In [ ]:
print(f"Runtime Agent: {launch_response.agent_id}")

#### AgentCore Runtime 삭제

In [ ]:
import os
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

delete_agent_response = agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_response.agent_id)
print(delete_agent_response)
print()

try:
    os.remove(".bedrock_agentcore.yaml")
    print("Successfully deleted local Runtime Agent config")
except Exception as e:
    print(f"Failed to delete local Agent config: {repr(e)}")

#### OAuth2 credential provider 삭제

In [ ]:
delete_credential_provider = agentcore_control_client.delete_oauth2_credential_provider(name=ms_provider["name"])
print(delete_credential_provider)